# 28 — Calibrate the detector and reproduce one saved 2B loop

Do not rerun Notebooks 25–27. Upload this notebook into a fresh Colab GPU runtime,
choose **Run all**, allow Drive, and enter the current balance and LIVE units/hour.
No repository clone, training, adapter upload, local setup, or code execution is needed.

**Problem:** the two simple questions finished; they did not exercise natural recovery.
**Goal:** check the detector against an actual saved training trace, then reproduce a
natural loop on its original question before evaluating RPMR.

Candidate: **Mbpp/568**, create a list of N empty dictionaries. Its saved 2B sample
repeats reconsideration text and consumes 4,096 thinking tokens without closing thinking.
The candidate and question were verified against saved training files; the ID is absent
from the current **464 evaluation IDs**. It is development data, not a new accuracy test.

The previous detector checked periods 8, 16 and 32 only. This diagnostic checks every
period from **8 to 128 tokens**, requiring at least **three consecutive identical blocks**
and at least **64 repeated tokens**. This is an existing exact-repeat detector design,
not the proposed novelty. It still does not identify every kind of semantic repetition.

First, in Colab, tokenize the saved historical trace and verify a trigger. Also verify
no trigger on the completed substring trace from Notebook 27, before loading model weights.
Historical trace tokens are reconstructed from saved text, not original generation IDs.
If calibration fails, the notebook stops and saves the report; it does not weaken rules.

Then generate the candidate from its question: **one greedy try**, followed only if needed
by **one sampled try** (temperature 0.6, top-p 0.95, top-k 20, seed 3407).
At most **4,096 total tokens per try**, stopping immediately when an exact thinking cycle
is found. No artificial prefix is inserted and no memory is edited.
The historical run used Unsloth and batch size 64; these new single-response trials use
the official pinned model revision. Historical sampling is not exactly reproducible here.

**Question:** can the calibrated detector identify a natural repeat on this candidate?
**Hypothesis:** at least one bounded try produces a detected thinking cycle.
**Change:** decoding mode, with question/model fixed. **Measure:** trigger, closure, token count,
time and estimated units. **Reference:** saved historical loop and completed non-loop trace.
**Data:** one known-loop development candidate. **Support:** a natural repeat event with
inspectable raw text. **Contradiction:** both tries finish or reach their caps without a trigger.
No repair efficacy, general loop rate, benchmark accuracy or novelty is established.

**Budget:** at most **three estimated units**, leaving the **20-unit reserve**.
At the previous 6.77 A100 units/hour this permits about **26.6 minutes**.
Planning estimate **4–12 minutes**, about **0.45–1.35 units at that rate**; assumes setup
similar to earlier runs and roughly 20 tokens/second. Detector overhead and longer prefixes
can change the time. Current balance/rate must be entered; prior allocation time is outside
the clock starting in cell 1. No extra retries or automatic recovery experiment follow.

Outputs: `report.json` and `answers.jsonl` under
`MyDrive/stop-overthinking/rpmr-candidate-search/<unique-run>/`, plus a downloadable ZIP.
Each response is saved immediately; the latest prefix is checkpointed every 64 tokens.
The notebook and watchdog attempt runtime disconnection. If they cannot, choose
**Runtime → Disconnect and delete runtime**. Entered-rate budgeting is not live billing.
**Prepared and syntax-checked, not run in Colab yet.**


## 1. Budget and Drive

Count setup time, protect the reserve, and save results in a new Drive folder.


In [ ]:
import sys, time, json, math, threading, subprocess, datetime, uuid
from pathlib import Path
from google.colab import drive, runtime

SESSION_START = time.monotonic()
SMOKE_CAP = 3.0
RESERVE = 20.0
BALANCE = float(input("Current Colab compute units available: "))
RATE = float(input("LIVE usage rate in compute units per hour, from Colab Resources: "))
if not math.isfinite(BALANCE) or not math.isfinite(RATE) or RATE <= 0 or BALANCE <= RESERVE:
    raise ValueError("Enter a finite positive rate and balance above the 20-unit reserve.")
SMOKE_CAP = min(SMOKE_CAP, BALANCE - RESERVE)
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = Path('/content/drive/MyDrive/stop-overthinking/rpmr-candidate-search') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
REPORT = dict(status='started', run_id=RUN_ID, model='Qwen/Qwen3.5-2B',
              initial_balance=BALANCE, entered_units_per_hour=RATE,
              estimated_unit_cap=SMOKE_CAP, reserve_units=RESERVE, checks=[],
              scope='saved-trace calibration and bounded candidate reproduction; no memory repair or accuracy')
LOCK = threading.RLock()
STOP_WATCHDOG = threading.Event()

def used_units():
    return (time.monotonic() - SESSION_START) / 3600 * RATE

def save_report():
    with LOCK:
        REPORT['wall_seconds'] = round(time.monotonic() - SESSION_START, 3)
        REPORT['estimated_units_used'] = round(used_units(), 4)
        temporary = OUT / 'report.json.tmp'
        temporary.write_text(json.dumps(REPORT, indent=2))
        temporary.replace(OUT / 'report.json')

def guard():
    if used_units() >= max(0, SMOKE_CAP - 0.1):
        raise TimeoutError('Estimated smoke-unit ceiling reached. Stop this session.')

def watchdog():
    while not STOP_WATCHDOG.wait(1):
        if used_units() >= max(0, SMOKE_CAP - 0.1):
            with LOCK:
                REPORT['status'] = 'budget_stopped'
                save_report()
            print('Budget ceiling: disconnecting runtime. Report saved to Drive.', flush=True)
            try:
                runtime.unassign()
            except Exception as exc:
                with LOCK:
                    REPORT['disconnect_error'] = str(exc)
                    save_report()
                print('Disconnect manually: Runtime → Disconnect and delete runtime.', flush=True)
            return

save_report()
threading.Thread(target=watchdog, daemon=True).start()
print(f'Smoke cap {SMOKE_CAP:.2f} units; reserve {RESERVE:.0f}. Results: {OUT}')
guard()


## 2. The embedded development example

Question and historical trace are embedded so nothing must be uploaded separately.
Provenance records the saved file hashes. No generated code or assertion is executed.


In [ ]:
CANDIDATE_INPUT = {'question': 'Write a function to create a list of N empty dictionaries.\n\nYour code must pass this test:\n```python\nassert empty_list(5)==[{},{},{},{},{}]\n```\n\nAnswer with one Python code block only, containing the complete function.', 'historical_trace': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Goal: Write a function to create a list of N empty dictionaries.\n    *   Constraint: The function must pass the specific test case: `assert empty_list(5)==[{},{},{},{},{}]`.\n    *   Output Format: One Python code block only, containing the complete function.\n\n2.  **Determine the Logic:**\n    *   The function needs to take an integer `N` as input.\n    *   It needs to return a list containing `N` empty dictionaries.\n    *   In Python, an empty dictionary is created using `{}`.\n    *   A list containing them would be `[]` with elements added.\n    *   A common way to do this is using a list comprehension or a loop.\n    *   Example: `def empty_list(n): return [{} for _ in range(n)]`\n\n3.  **Verify the Logic:**\n    *   Input: `5`\n    *   `range(5)` produces `0, 1, 2, 3, 4`.\n    *   List comprehension `[{} for _ in range(5)]` produces `[{}, {}, {}, {}, {}]`.\n    *   This matches the expected output `[{},{},{},{},{}]`.\n    *   Python\'s `dict()` constructor can also be used, but `{} is the standard way for empty dicts.\n\n4.  **Check Constraints:**\n    *   "Your code must pass this test".\n    *   "Answer with one Python code block only".\n\n5.  **Draft the Code:**\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n\n    # Optional: Include the assertion in the code block or just the function?\n    # The prompt says "Answer with one Python code block only, containing the complete function."\n    # It doesn\'t explicitly forbid adding the assertion, but usually, it means just the function definition or the test code.\n    # However, looking at the instruction "Your code must pass this test", I should probably just define the function and maybe include the test call inside the block to demonstrate it works, OR just the function.\n    # Usually, when asked for a function to pass a test, providing the function is the core requirement.\n    # But to be safe and ensure it passes the test *in the context of the user\'s prompt*, I should define the function.\n    # Wait, looking at the instruction "Answer with one Python code block only".\n    # I will provide the function definition.\n    ```\n\n    Let\'s refine the function to be robust.\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    Is there any edge case? `n=0`? Returns `[]`. Correct. `n=1`? Returns `[{}]`. Correct.\n    The test case specifically calls `empty_list(5)`.\n\n6.  **Final Review:**\n    *   Does the code block contain only the function?\n    *   Does it solve the problem?\n    *   Is it one code block?\n\n    Yes.\n\n    Wait, I should check if the user wants the test assertion included in the code block. The prompt says "Your code must pass this test". Often, this implies the function itself is what matters. However, sometimes providing the test call inside the block helps verify. But the instruction says "Answer with one Python code block only, containing the complete function."\n    So I will write the function. I don\'t need to write `assert ...` inside the function definition, just the function definition.\n\n    Let\'s write it clearly.\n\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n\n    Wait, checking the test again: `assert empty_list(5)==[{},{},{},{},{}]`.\n    This implies the function name is `empty_list`.\n    I will include the function definition.\n\n7.  **Final Output Generation:**\n    Create the markdown code block.\n\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n\n    Wait, I need to make sure I don\'t add any extra text outside the code block. The prompt says "Answer with one Python code block only".\n\n    Okay, ready.\n\n    One more check: `assert empty_list(5)==[{},{},{},{},{}]`.\n    The function needs to be named `empty_list`.\n    The argument is `n`.\n    The return type is `list` of `dict`.\n\n    Code:\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    This looks correct.\n\n    Wait, is there any ambiguity about "one Python code block only"?\n    It means I shouldn\'t put explanations outside. Just the code block.\n\n    Okay.\n\n    Wait, one detail. Does the user want the `assert` statement to be part of the output?\n    "Your code must pass this test" -> The function must satisfy the assertion.\n    "Answer with one Python code block only, containing the complete function."\n    So the content of the block is the function.\n\n    Final check on constraints:\n    - Function name: `empty_list`\n    - Argument: `n` (implied by the test `empty_list(5)`)\n    - Return: List of N empty dicts.\n    - Code block: Only.\n\n    Okay.\n\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    This is sufficient.\n\n    Wait, I should double check if there are any imports needed. No.\n    Is `range` available? Yes.\n    Is `dict` available? Yes.\n    Is `list` available? Yes.\n\n    Okay, proceeding.\n\n    Wait, I need to make sure I don\'t accidentally add comments that might violate "one Python code block only" if interpreted strictly, but comments are allowed in Python code blocks. However, usually "one code block" means no markdown text outside.\n    I will keep it clean.\n\n    Wait, looking at the instruction again: "Answer with one Python code block only".\n    This means I should not write any introductory text like "Here is the code:".\n    Just the code block.\n\n    Okay.\n\n    Wait, one detail in the test: `assert empty_list(5)==[{},{},{},{},{}]`.\n    This is Python syntax.\n    The function should be defined.\n\n    Okay.\n\n    Wait, is there any possibility that the user wants the assertion to be part of the code block?\n    "Answer with one Python code block only, containing the complete function."\n    This implies the function is the content. The assertion is the condition for the function to be valid.\n    So I will just write the function.\n\n    Wait, I\'ll add the function definition.\n\n    Wait, one more thing. The prompt says "Write a function".\n    So the code block should define the function.\n\n    Okay.\n\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    This is correct.\n\n    Wait, I\'ll check if `n` needs to be validated.\n    Usually, `n` is assumed to be non-negative.\n    The test uses `5`.\n    The logic holds.\n\n    Final decision:\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    Wait, I should check if there\'s any specific naming convention required. "empty_list". Yes.\n    Argument "n". Yes.\n\n    Okay.\n\n    Wait, I need to ensure I don\'t include any markdown outside the code block.\n    I will output just the code block.\n\n    Wait, I\'ll check if I should include the test assertion in the code block to demonstrate it works?\n    "Your code must pass this test".\n    "Answer with one Python code block only, containing the complete function."\n    This implies the function itself.\n    I will stick to the function definition.\n\n    Okay.\n    Wait, one more check.\n    Is `n` an integer? Yes.\n    Is `range(n)` valid for negative numbers? In Python 3, `range(-5)` is `range(0, -5)`, which is empty.\n    So `[]` is returned.\n    But the test uses `5`.\n    So it\'s fine.\n\n    Okay, ready to write.\n\n    Wait, I should check if the user wants the assertion to be inside the function? No, that doesn\'t make sense.\n    The assertion is the test case.\n    So the function is the answer.\n\n    Okay.\n\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    Wait, I\'ll check if I need to import anything. No.\n    Okay.\n\n    Wait, I\'ll check if I should handle the case where `n` is not provided?\n    The function signature is `empty_list(n)`.\n    So `n` is required.\n\n    Okay.\n    Wait, I\'ll check if I should put the test case inside the function?\n    No, that\'s not a function to create a list. That\'s a function to run a test.\n    So just the function.\n\n    Okay.\n    Wait, I\'ll check if I should use `dict()` instead of `{}`.\n    `{}` is the standard way. `dict()` returns a new dict.\n    `{}` is more concise.\n\n    Okay.\n    Wait, I\'ll check if I should use `list` comprehension.\n    Yes, `[{} for _ in range(n)]`.\n    This is the most efficient and readable way.\n\n    Okay.\n    Wait, I\'ll check if I should add a docstring?\n    Not required by the prompt, but good practice.\n    I\'ll skip it to keep it minimal and safe.\n\n    Okay.\n    Wait, I\'ll check if I should add comments.\n    Not required.\n    Okay.\n\n    Final check.\n    Input: `empty_list(5)`\n    Output: `[{},{},{},{},{}]`\n    Code:\n    ```python\n    def empty_list(n):\n        return [{} for _ in range(n)]\n    ```\n    This passes the assertion.\n\n    Okay.\n    Wait, I\'ll check if I need to put the function inside the block.\n    Yes.\n    Okay.\n\n    Wait, I\'ll check if I should put the test assertion in the code block?\n    "Answer with one Python code block only, containing the complete function."\n    This means the content of the block is the function.\n    Okay.\n    Wait, I\'ll check if I should add the assertion to the top of the file?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add `if __name__ == "__main__":`?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should include the test assertion in the code block to verify the user\'s requirement?\n    "Your code must pass this test".\n    This is a constraint on the function.\n    The answer is the function.\n    Okay.\n\n    Wait, I\'ll check if I should include the assertion in the code block?\n    "Answer with one Python code block only, containing the complete function."\n    Okay, I will just provide the function.\n\n    Wait, I\'ll check if I should add the test assertion to the code block to show it works?\n    No, the prompt says "Answer with one Python code block only, containing the complete function."\n    So the function is the answer.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block to verify the user\'s requirement?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n\n    Wait, I\'ll check if I should add the test assertion to the code block?\n    No.\n    Okay.\n    Wait, I\'ll check if I should add the test assertion to the code', 'negative_tokens': [1919, 369, 264, 11088, 3377, 421, 628, 381, 27050, 1608, 279, 32713, 3136, 14334, 13, 6558, 728, 1683, 1472, 279, 5313, 25, 271, 16, 13, 2972, 3406, 1414, 11256, 92217, 1368, 279, 1414, 369, 4147, 11, 460, 220, 15, 198, 17, 13, 2972, 87855, 5587, 92217, 1165, 1144, 310, 3627, 279, 1483, 2234, 314, 1754, 3542, 321, 958, 264, 32713, 3136, 310, 1423, 279, 21354, 37172, 1973, 40922, 271, 760, 32713, 3136, 5313, 25, 198, 12, 5272, 1330, 26564, 318, 2282, 321, 1245, 8, 310, 6762, 279, 1428, 3136, 198, 12, 13262, 3627, 314, 279, 1483, 2234, 314, 1754, 3542, 198, 12, 3095, 567, 12460, 264, 11173, 3542, 11, 3166, 279, 2047, 7210, 2980, 567, 1423, 279, 1118, 30520, 314, 421, 3542, 198, 12, 5376, 279, 6966, 2984, 1669, 271, 9764, 728, 3165, 264, 15352, 6093, 440, 25, 198, 16, 13, 561, 709, 7873, 198, 17, 13, 68868, 314, 279, 5313, 198, 18, 13, 10041, 1105, 11256, 198, 19, 13, 12996, 10121, 271, 71093, 12305, 198, 727, 21354, 20567, 5062, 889, 4956, 7008, 1590, 198, 262, 413, 524, 1414, 25, 198, 285, 460, 220, 15, 6987, 262, 653, 10153, 310, 3436, 279, 1483, 2234, 314, 1754, 3542, 198, 262, 1122, 3442, 283, 4535, 198, 262, 1866, 4956, 283, 220, 15, 198, 262, 2047, 283, 220, 15, 6987, 262, 364, 1245, 11, 1122, 303, 12867, 7008, 1590, 198, 285, 653, 1368, 3542, 369, 2582, 303, 279, 1428, 3136, 198, 285, 413, 1122, 303, 1122, 3442, 321, 1122, 3442, 89506, 60, 2518, 2047, 25, 198, 309, 653, 14142, 2047, 7210, 310, 279, 2234, 1238, 279, 1483, 30520, 198, 309, 2047, 283, 1122, 3442, 89506, 60, 478, 220, 16, 8675, 285, 653, 5376, 279, 1483, 2234, 314, 279, 1428, 3542, 198, 285, 1122, 3442, 89506, 60, 283, 1245, 8675, 285, 653, 5376, 279, 6966, 2984, 198, 285, 1866, 4956, 283, 1866, 8474, 4956, 11, 1245, 471, 2047, 478, 220, 16, 8, 198, 1031, 262, 460, 1866, 4956, 198, 71093, 271, 13784, 11, 353, 1144, 310, 37781, 411, 13, 561, 4125, 369, 421, 948, 567, 12460, 264, 11173, 3542, 11, 567, 1144, 310, 3166, 279, 2047, 7210, 310, 279, 2234, 1238, 279, 1483, 30520, 314, 421, 3542, 13, 1921, 567, 1048, 1144, 310, 3579, 279, 1105, 1332, 279, 3542, 7701, 5081, 2942, 303, 279, 1428, 3136, 13, 271, 9764, 728, 1683, 1495, 440, 264, 2577, 5313, 25, 271, 71093, 12305, 198, 727, 21354, 20567, 5062, 889, 4956, 7008, 1590, 198, 262, 413, 524, 1414, 25, 198, 285, 460, 220, 15, 6987, 262, 653, 10153, 310, 3436, 279, 1483, 2234, 314, 1754, 3542, 198, 262, 1122, 3442, 283, 4535, 198, 262, 1866, 4956, 283, 220, 15, 198, 262, 2047, 283, 220, 15, 6987, 262, 364, 1245, 11, 1122, 303, 12867, 7008, 1590, 198, 285, 653, 1368, 3542, 369, 2582, 303, 279, 1428, 3136, 198, 285, 413, 1122, 303, 1122, 3442, 321, 1122, 3442, 89506, 60, 2518, 2047, 25, 198, 309, 653, 14142, 2047, 7210, 310, 279, 2234, 1238, 279, 1483, 30520, 198, 309, 2047, 283, 1122, 3442, 89506, 60, 478, 220, 16, 8675, 285, 653, 5376, 279, 1483, 2234, 314, 279, 1428, 3542, 198, 285, 1122, 3442, 89506, 60, 283, 1245, 8675, 285, 653, 5376, 279, 6966, 2984, 198, 285, 1866, 4956, 283, 1866, 8474, 4956, 11, 1245, 471, 2047, 478, 220, 16, 8, 198, 1031, 262, 460, 1866, 4956, 198, 71093, 271, 40, 2688, 14419, 279, 1483, 2234, 314, 1754, 3542, 310, 5790, 874, 40922, 2785, 279, 1428, 3136, 13, 561, 32713, 3136, 14334, 8346, 28750, 1423, 279, 21354, 37172, 1973, 11173, 5587, 13, 271, 760, 1328, 369, 19720, 264, 8476, 3136, 421, 50110, 321, 16577, 3018, 383, 3542, 54133, 13, 3113, 14419, 279, 1483, 2234, 314, 1754, 3542, 11, 567, 628, 5964, 7995, 279, 2610, 3136, 22005, 13, 271, 4350, 264, 21814, 3542, 369, 1669, 11, 567, 7265, 279, 2047, 7210, 310, 21019, 279, 3555, 30520, 11, 21873, 874, 11173, 5587, 2905, 2785, 279, 1428, 3136, 13, 1061, 5313, 5954, 586, 310, 28750, 10724, 279, 6966, 4752, 37172, 2984, 13, 198, 248069, 271, 8160, 579, 264, 12654, 709, 421, 55288, 279, 2984, 314, 279, 21354, 64258, 37172, 440, 874, 11173, 5587, 11, 3055, 440, 264, 11346, 15673, 314, 1204, 424, 13081, 6607, 4886, 321, 11173, 5587, 13, 271, 71093, 12305, 198, 727, 21354, 20567, 5062, 889, 4956, 7008, 1590, 198, 262, 4071, 198, 262, 5019, 279, 2984, 314, 279, 21354, 64258, 37172, 440, 874, 11173, 5587, 13, 271, 262, 17167, 25, 198, 285, 1414, 318, 485, 1590, 5396, 886, 310, 22903, 13, 271, 262, 5019, 25, 198, 285, 514, 25, 16777, 314, 279, 21354, 37172, 440, 874, 11173, 5587, 13, 198, 262, 4071, 198, 262, 413, 524, 1414, 25, 198, 285, 460, 220, 15, 271, 262, 1122, 3442, 283, 4535, 198, 262, 1866, 4956, 283, 220, 15, 198, 262, 2047, 283, 220, 15, 271, 262, 364, 1245, 11, 1122, 303, 12867, 7008, 1590, 198, 285, 413, 1122, 303, 1122, 3442, 321, 1122, 3442, 89506, 60, 2518, 2047, 25, 198, 309, 2047, 283, 1122, 3442, 89506, 60, 478, 220, 16, 271, 285, 1122, 3442, 89506, 60, 283, 1245, 198, 285, 1866, 4956, 283, 1866, 8474, 4956, 11, 1245, 471, 2047, 478, 220, 16, 8, 271, 262, 460, 1866, 4956, 198, 71093, 271, 13962, 2500, 1049, 20457, 271, 16, 13, 2972, 3406, 2824, 53844, 92217, 1368, 279, 1879, 886, 369, 4147, 11, 279, 709, 6849, 4523, 220, 15, 13, 198, 17, 13, 2972, 7207, 6369, 13249, 74963, 92217, 198, 256, 471, 357, 1510, 1700, 3442, 63, 10675, 13518, 279, 1483, 3753, 2234, 314, 1754, 3542, 13, 198, 256, 471, 357, 1510, 2282, 63, 7210, 14973, 279, 1151, 314, 279, 1428, 3136, 13, 198, 256, 471, 1577, 279, 1510, 1246, 63, 7210, 10480, 1472, 279, 886, 11, 413, 264, 3542, 369, 2582, 303, 279, 1428, 3136, 318, 72, 1673, 2487, 1141, 1483, 2234, 369, 60052, 1510, 2282, 63, 681, 279, 1510, 2282, 63, 7210, 369, 10530, 310, 1066, 1238, 279, 1483, 30520, 314, 421, 3542, 13, 198, 256, 471, 561, 2984, 314, 279, 1428, 3136, 369, 5860, 1608, 1510, 1246, 471, 2047, 478, 220, 16, 7561, 321, 279, 6966, 2984, 369, 31961, 13, 198, 18, 13, 2972, 1409, 321, 11156, 68230, 92217, 198, 256, 471, 3984, 25, 496, 1393, 681, 1332, 307, 369, 279, 2984, 314, 279, 1879, 886, 13, 198, 256, 471, 11156, 25, 496, 5781, 681, 1332, 580, 369, 279, 1324, 314, 4752, 5587, 13, 271, 13962, 12996, 23791, 271, 71093, 12305, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 13290, 13290, 5876, 2673, 220, 653, 8984, 25, 220, 18, 3376, 13290, 871, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 5876, 51365, 2673, 257, 653, 8984, 25, 220, 16, 3376, 65, 871, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 79, 1207, 74, 360, 2673, 262, 653, 8984, 25, 220, 18, 3376, 86, 432, 871, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 2673, 670, 653, 8984, 25, 220, 15, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 39840, 2673, 262, 653, 8984, 25, 220, 21, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 64, 2673, 260, 653, 8984, 25, 220, 16, 198, 1302, 12265, 467, 20567, 5062, 889, 4956, 437, 365, 565, 11663, 2673, 256, 653, 8984, 25, 220, 18, 3376, 11663, 871, 198, 71093, 271, 13962, 5141, 20495, 778, 271, 12, 2972, 87855, 43038, 92217, 561, 709, 24546, 421, 874, 3542, 7701, 777, 1056, 2957, 2785, 279, 1428, 3136, 13, 198, 12, 2972, 3406, 40275, 92217, 8274, 806, 20335, 310, 5471, 1857, 5787, 13, 198, 12, 2972, 10570, 15174, 40275, 92217, 5019, 220, 16, 11, 430, 3481, 13, 198, 12, 2972, 6375, 3593, 18034, 92217, 85943, 391, 34337, 279, 21354, 37172, 1973, 40922, 13, 271, 1919, 6093, 369, 22113, 364, 1379, 14542, 958, 4886, 321, 13081, 660, 6607, 4886, 21094, 391, 13, 248046, 198, 248044], 'provenance': {'task_id': 'Mbpp/568', 'source': 'existing MBPP training-generation diagnostic', 'source_file': 'results/2026-09-24-thesis-run/train-mbpp.jsonl', 'source_file_sha256': '05b7d718ed97c65db07d138e7fa909e197936f88d2cb4c7f453fa18681e90f7b', 'question_file': 'results/2026-09-24-thesis-run/train-set-mbpp.jsonl', 'question_file_sha256': 'eb32b6974c756f28695789ad24e7698d984e44f879b56f1844010d77aef509c3', 'checked_absent_from_current_464_evaluation_ids': True, 'current_evaluation_id_count': 464, 'original_model': 'unsloth/Qwen3.5-2B', 'original_batch_size': 64, 'original_seed': 3407, 'original_thinking_tokens': 4096, 'original_thinking_closed': False}}
QUESTION = CANDIDATE_INPUT["question"]
REPORT["candidate_provenance"] = CANDIDATE_INPUT["provenance"]
save_report()
print("Candidate:", REPORT["candidate_provenance"]["task_id"])


## 3. Calibration, at most two tries, save, and disconnect

Requires a trigger on the stored loop and no trigger on the stored completed thinking trace.
Then starts from the question normally. Greedy success stops the run without sampling.
If no live loop is found, the report is explicitly inconclusive for recovery.


In [ ]:
try:
    guard()
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'transformers==5.18.0','accelerate','safetensors'],check=True,
                   timeout=max(1,(SMOKE_CAP-used_units()-0.1)/RATE*3600))
    guard()
    import os,torch,transformers
    from transformers import AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError('Choose a fresh Colab GPU runtime.')
    torch.manual_seed(3407)
    torch.cuda.manual_seed_all(3407)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    MODEL_ID='Qwen/Qwen3.5-2B'
    REVISION='15852e8c16360a2fea060d615a32b45270f8a8fc'
    REPORT['model_revision']=REVISION
    REPORT['environment']=dict(python=sys.version,torch=torch.__version__,
        transformers=transformers.__version__,gpu=torch.cuda.get_device_name(0),model_dtype=str(dtype),seed=3407)
    REPORT['settings']=dict(max_total_new_tokens_per_try=4096,max_tries=2,period_min=8,period_max=128,
        min_consecutive_blocks=3,min_repeated_tokens=64,memory_repair_enabled=False,
        code_execution=False,sampled_settings=dict(temperature=0.6,top_p=0.95,top_k=20))
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
    end_ids=tokenizer.encode('</think>',add_special_tokens=False)
    def thinking_prefix(ids):
        for i in range(len(ids)-len(end_ids)+1):
            if ids[i:i+len(end_ids)]==end_ids:
                return ids[:i]
        return ids
    def repeat_event(prefix):
        for period in range(8,129):
            repeats=max(3,math.ceil(64/period))
            span=period*repeats
            if len(prefix)<span:
                continue
            last=prefix[-period:]
            if all(prefix[-(j+1)*period:-j*period]==last for j in range(1,repeats)):
                return dict(at_token=len(prefix),period=period,repeats=repeats,repeated_tokens=span,
                            repeated_block_token_ids=last)
        return None
    def first_event(ids):
        for end in range(64,len(ids)+1):
            if end%64==0:
                guard()
            event=repeat_event(ids[:end])
            if event is not None:
                return event
        return None
    historical=tokenizer.encode(CANDIDATE_INPUT['historical_trace'],add_special_tokens=False)
    positive=first_event(thinking_prefix(historical))
    negative=first_event(thinking_prefix(CANDIDATE_INPUT['negative_tokens']))
    REPORT['calibration']=dict(historical_tokens_reconstructed=True,
        historical_repeat=positive,completed_substring_repeat=negative)
    save_report()
    if positive is None:
        raise RuntimeError('Detector did not recognize the saved loop. Stop before downloading model weights.')
    if negative is not None:
        raise RuntimeError('Detector flagged the completed substring trace. Review before live generation.')
    REPORT['checks'].append('Saved positive/negative detector calibration passed in Colab')
    save_report()
    guard()
    loader=getattr(transformers,'AutoModelForMultimodalLM',None)
    if loader is None:
        loader=transformers.AutoModelForImageTextToText
    model=loader.from_pretrained(MODEL_ID,revision=REVISION,dtype=dtype,
                                 device_map={'':0},attn_implementation='sdpa')
    model.eval()
    eos=model.generation_config.eos_token_id
    eos_ids=set(eos if isinstance(eos,list) else [eos])
    responses=[]
    def choose_token(logits,mode,generator):
        scores=logits[0,-1].float()
        if mode=='greedy':
            return scores.argmax().reshape(1,1)
        scores=scores/0.6
        cutoff=torch.topk(scores,20).values[-1]
        scores=scores.masked_fill(scores<cutoff,float('-inf'))
        ordered,indices=torch.sort(scores,descending=True)
        cumulative=torch.softmax(ordered,dim=-1).cumsum(dim=-1)
        remove=cumulative>0.95
        remove[1:]=remove[:-1].clone()
        remove[0]=False
        ordered=ordered.masked_fill(remove,float('-inf'))
        picked=torch.multinomial(torch.softmax(ordered,dim=-1),1,generator=generator)
        return indices[picked].reshape(1,1)
    for mode in ('greedy','sampled'):
        guard()
        prompt=tokenizer.apply_chat_template([dict(role='user',content=QUESTION)],
            tokenize=False,add_generation_prompt=True,enable_thinking=True)
        ids=tokenizer(prompt,return_tensors='pt').input_ids.to('cuda')
        generator=torch.Generator(device='cuda').manual_seed(3407)
        with torch.no_grad():
            output=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=True,logits_to_keep=1)
        cache=output.past_key_values
        cursor=choose_token(output.logits,mode,generator)
        tokens=[]
        event=None
        in_thinking=True
        stop_reason='total_token_cap'
        torch.cuda.synchronize()
        started=time.monotonic()
        with torch.no_grad():
            for step in range(4096):
                guard()
                token=int(cursor.item())
                tokens.append(token)
                text=tokenizer.decode(tokens,skip_special_tokens=False)
                if '</think>' in text:
                    in_thinking=False
                event=repeat_event(tokens) if in_thinking else None
                if event is not None:
                    stop_reason='exact_thinking_repeat_detected'
                elif token in eos_ids:
                    stop_reason='eos'
                if len(tokens)%64==0 or event is not None or token in eos_ids:
                    with LOCK:
                        REPORT['latest_prefix']=dict(mode=mode,generated_tokens=len(tokens),token_ids=list(tokens),
                            raw_output=text,thinking_closed=not in_thinking,event=event)
                        save_report()
                if event is not None or token in eos_ids:
                    break
                output=model(input_ids=cursor,
                    attention_mask=torch.ones((1,ids.shape[1]+len(tokens)),device='cuda',dtype=torch.long),
                    past_key_values=cache,use_cache=True,logits_to_keep=1)
                if not torch.isfinite(output.logits).all().item():
                    raise RuntimeError('Non-finite logits.')
                cache=output.past_key_values
                cursor=choose_token(output.logits,mode,generator)
        torch.cuda.synchronize()
        seconds=time.monotonic()-started
        answer=dict(task_id='Mbpp/568',question=QUESTION,mode=mode,seed=3407,
            generated_tokens=len(tokens),token_ids=tokens,raw_output=tokenizer.decode(tokens,skip_special_tokens=False),
            thinking_closed=not in_thinking,first_repeat_event=event,stop_reason=stop_reason,
            decode_seconds=seconds,tokens_per_second=len(tokens)/max(seconds,1e-9),
            artificial_prefix_inserted=False,memory_edits=0)
        with (OUT/'answers.jsonl').open('a') as stream:
            stream.write(json.dumps(answer)+'\n')
            stream.flush()
            os.fsync(stream.fileno())
        responses.append(answer)
        REPORT['completed_tries']=len(responses)
        REPORT['response_summary']=[{k:x[k] for k in ('mode','generated_tokens','thinking_closed',
            'first_repeat_event','stop_reason','decode_seconds','tokens_per_second')} for x in responses]
        save_report()
        print(f'{mode}: {len(tokens)} tokens, {seconds:.1f}s, {stop_reason}',flush=True)
        del cache,output
        torch.cuda.empty_cache()
        if event is not None:
            break
    found=any(x['first_repeat_event'] is not None for x in responses)
    REPORT['status']='natural_repeat_reproduced' if found else 'calibration_passed_no_live_repeat'
    REPORT['interpretation']=('Natural candidate repeat reproduced. Review the saved text before a paired recovery test.' if found
        else 'Archived detector calibration passed, but bounded new tries did not reproduce a repeat. Do not claim recovery evidence.')
    REPORT['accuracy_measured']=False
    REPORT['repairs_applied']=0
    REPORT['peak_allocated_gpu_memory_gb']=torch.cuda.max_memory_allocated()/1e9
    REPORT.pop('latest_prefix',None)
    guard()
except Exception as exc:
    with LOCK:
        if REPORT.get('status')!='budget_stopped':
            REPORT['status']='failed'
        REPORT['error']=f'{type(exc).__name__}: {exc}'
    import traceback
    traceback.print_exc()
finally:
    save_report()
    console={k:v for k,v in REPORT.items() if k!='latest_prefix'}
    print(json.dumps(console,indent=2))
    print('Drive report:',OUT/'report.json')
    try:
        from google.colab import files
        import zipfile
        archive=Path('/content')/f'rpmr-candidate-{RUN_ID}.zip'
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zipped:
            for path in (OUT/'report.json',OUT/'answers.jsonl'):
                if path.exists():
                    zipped.write(path,path.name)
        files.download(str(archive))
    except Exception as exc:
        print('Download unavailable; files remain on Drive:',exc)
    try:
        runtime.unassign()
    except Exception as exc:
        REPORT['disconnect_error']=str(exc)
        save_report()
        print('Disconnect manually: Runtime → Disconnect and delete runtime.')
    else:
        STOP_WATCHDOG.set()
